# 04. RAG 파이프라인 구축

**목적**: 의학 가이드라인 PDF를 청크로 분리하고 벡터 DB에 저장

**입력 문서**
- 2023 당뇨병 진료지침 (대한당뇨병학회)
- 2022 고혈압 진료지침 (대한고혈압학회)
- 2025 한국인 영양소 섭취기준 (보건복지부·한국영양학회)

**출력**: ChromaDB 벡터 DB

## 0. 패키지 설치 및 라이브러리 로드

In [ ]:
!pip install -q langchain langchain-community langchain-openai chromadb pymupdf tiktoken

In [ ]:
import os
import warnings
from google.colab import drive, userdata

from langchain.document_loaders import PyMuPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain.vectorstores import Chroma

warnings.filterwarnings('ignore')

drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/AH_03_06'

In [ ]:
# OpenAI API 키 설정
# Colab 왼쪽 사이드바 자물쇠 아이콘(Secrets)에 OPENAI_API_KEY 등록 필요
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

## 1. PDF 로드

In [ ]:
PDF_FILES = {
    '당뇨병진료지침': f'{BASE}/data/raw/pdf/2023_당뇨병_진료지침_전문_240620.pdf',
    '고혈압진료지침': f'{BASE}/data/raw/pdf/13.대한고혈압학회_-_2022_제5판_고혈압_진료지침_221206.pdf',
    '영양소섭취기준': f'{BASE}/data/raw/pdf/웹용_2025_KDRI-국문_요약본.pdf',
}

all_docs = []

for source, path in PDF_FILES.items():
    loader = PyMuPDFLoader(path)
    docs   = loader.load()
    for doc in docs:
        doc.metadata['source'] = source
    all_docs.extend(docs)
    print(f'{source}: {len(docs)}페이지 로드 완료')

print(f'\n총 {len(all_docs)}페이지 로드 완료')

## 2. 청크 분리

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=['\n\n', '\n', '. ', ' ', ''],
)

chunks = text_splitter.split_documents(all_docs)
print(f'총 청크 수: {len(chunks):,}개')

# 소스별 청크 수 확인
from collections import Counter
source_counts = Counter(chunk.metadata['source'] for chunk in chunks)
for source, count in source_counts.items():
    print(f'  {source}: {count:,}개')

In [ ]:
# 청크 샘플 확인
print('=== 청크 샘플 ===')
for i in [0, 100, 200]:
    print(f'\n[청크 {i}] ({chunks[i].metadata["source"]})')
    print(chunks[i].page_content[:200])

## 3. 벡터 임베딩 및 ChromaDB 저장

In [ ]:
CHROMA_PATH = f'{BASE}/data/chroma_db'
os.makedirs(CHROMA_PATH, exist_ok=True)

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=CHROMA_PATH,
)

vectordb.persist()
print(f'벡터 DB 저장 완료: {CHROMA_PATH}')
print(f'저장된 벡터 수: {vectordb._collection.count():,}개')

## 4. 검색 테스트

In [ ]:
# 그룹별 검색 키워드 정의
GROUP_SEARCH_KEYWORDS = {
    '정상군':           '성인 균형 식단 권장 영양소',
    '고혈압군':          '고혈압 나트륨 제한 식사요법',
    '혈당이상군':         '당뇨 전단계 탄수화물 제한 식사요법',
    '비만군':            '비만 칼로리 제한 체중 감량 식단',
    '고혈압+혈당이상군':   '고혈압 당뇨 복합 식사요법 나트륨 탄수화물',
    '고혈압+비만군':      '고혈압 비만 칼로리 나트륨 제한',
    '혈당이상+비만군':    '당뇨 비만 저탄수화물 저칼로리 식단',
    '복합위험군':         '당뇨 고혈압 비만 복합 치료 식이요법',
}

def search_rag(group, k=3):
    keyword = GROUP_SEARCH_KEYWORDS.get(group, '균형 식단')
    results = vectordb.similarity_search(keyword, k=k)
    return results

# 검색 테스트
print('=== 혈당이상군 검색 결과 ===')
results = search_rag('혈당이상군')
for i, doc in enumerate(results):
    print(f'\n[{i+1}] ({doc.metadata["source"]})')
    print(doc.page_content[:200])

## 5. 결과 요약

In [ ]:
print('=' * 60)
print('RAG 파이프라인 구축 결과 요약')
print('=' * 60)
print(f'로드 문서  : {len(PDF_FILES)}개')
print(f'총 청크 수 : {len(chunks):,}개')
print(f'벡터 DB    : {CHROMA_PATH}')
print()
print('→ 다음 단계: 05_llm_diet_guide.ipynb')
print('=' * 60)